# 09 -- Performance Analysis and Report (explicitly labeled research demonstration)

> **This notebook injects `FakeEstimator`/`FakeHMMFitter` (deterministic, non-fitting stand-ins) instead of scikit-learn's `HistGradientBoostingClassifier` / hmmlearn's `GaussianHMM`.** This is the explicitly-labeled research-demonstration case this project's own constraints reserve for fake fitters -- it exists solely to show Stage 7/8/9's wiring end-to-end without needing the optional dependencies installed. **The resulting report's `reproducibility_status` is set to `NOT_RUN` and every metric in it is meaningless** -- never compare it to `report_current.html` as if it were a real result.

In [ ]:
import atlas_quant
print("atlas_quant path:", atlas_quant.__file__)

import sys
sys.path.insert(0, ".")  # this notebook's own directory (for _fixtures.py) -- not a legacy path hack
import _fixtures


In [ ]:
# Test-only fakes, used here only because this cell is an explicitly labeled
# research demonstration -- never used this way in atlas_quant.cli or production code.
import sys
sys.path.insert(0, "../../../../tests")
from fixtures.filing_momentum_ml import FakeEstimator, FakeHMMFitter


In [ ]:
from atlas_quant.backtest.filing_momentum_runner import FilingMomentumBacktestConfig
from atlas_quant.strategies.filing_momentum_ml.estimator import EstimatorBuildInfo
from atlas_quant.strategies.filing_momentum_ml.performance_analysis import analyze_backtest_result
from atlas_quant.strategies.filing_momentum_ml.production.orchestration import (
    ProductionRunInputs, run_filing_momentum_production_backtest,
)
import atlas_quant.strategies.filing_momentum_ml.production.orchestration as orchestration_module
from atlas_quant.strategies.filing_momentum_ml.reporting.report_model import ReportOptions
from atlas_quant.strategies.filing_momentum_ml.sector_encoding import SectorEncoder
from atlas_quant.reporting.domain import ReproducibilityStatus

def _fake_estimator_factory(model_config):
    return FakeEstimator(), EstimatorBuildInfo(
        estimator_type="FakeEstimator", parameters={}, library="test-fake", library_version=None,
    )

# Monkeypatch this notebook process only -- never done in atlas_quant.cli or any production path.
orchestration_module.missing_required_for_production = lambda report: ()
orchestration_module.build_hgbc_estimator = _fake_estimator_factory
orchestration_module.HmmlearnFitter = lambda: FakeHMMFitter()


In [ ]:
bundle = _fixtures.build_synthetic_bundle()
calendar = _fixtures.build_trading_calendar(bundle)
periods = _fixtures.build_periods()
manifest = _fixtures.build_synthetic_manifest()

inputs = ProductionRunInputs(
    backtest_config=FilingMomentumBacktestConfig(), periods=periods, universe=bundle.universe,
    benchmark_instrument_id=bundle.benchmark_instrument_id, trading_calendar=calendar,
    sector_encoder=SectorEncoder(), filings_by_instrument=bundle.filings_by_instrument,
    prices_by_instrument=bundle.prices_by_instrument, sector_by_instrument=bundle.sector_by_instrument,
    manifest=manifest, report_options=ReportOptions(include_source_comparison=False),
    reproducibility_status=ReproducibilityStatus.NOT_RUN,
)
result = run_filing_momentum_production_backtest(inputs)
print("state:", result.state.value)
print("completed quarters:", result.backtest_result.completed_quarter_count if result.backtest_result else None)
if result.performance_analysis is not None:
    print("overall total return (SYNTHETIC, MEANINGLESS):", result.performance_analysis.overall.metrics)
if result.report is not None:
    print("report reproducibility_status:", result.report.metadata.reproducibility_status.value)


## Findings

- With the dependency gate and estimator/fitter faked, the full Stage 7 -> 8 -> 9 pipeline runs to completion and produces a structurally valid `FilingMomentumReport`.

## Limitations

- **Every number this cell prints is synthetic-input-derived and meaningless.** `reproducibility_status` is explicitly `NOT_RUN`. This notebook demonstrates wiring, not a strategy result -- see `docs/reproducibility_findings.md` for what a genuine reproduction attempt would require and why it has not been performed.